# 데이터구조 11주차 실습 — 힙과 힙 정렬 (학생용)

**담당:** 임화정 | **환경:** Python 3.8

## 실습 목표
- 최소 힙 / 최대 힙 `BinaryHeap`을 구현한다.
- 제자리 힙 정렬 `heapsort_inplace`를 구현한다.
- `heapq`로 우선순위 큐와 Top-k를 구현한다.
- 복잡도를 실험으로 확인하고 해석한다.

**규칙:** `# TODO` 부분을 위에서 아래로 구현하고, 각 셀의 `assert` 테스트를 통과시키세요. 입력 리스트를 slicing하거나 `sorted()`로 우회하지 마세요(정렬 실습 셀 제외).

## 1. BinaryHeap 구현 (최소 힙 / 최대 힙)

- `mode`는 `'min'` 또는 `'max'`만 허용합니다. 그 외는 `ValueError`.
- 내부 배열 `self._data`를 사용하고, `data` 속성은 **복사본**을 반환합니다.
- 빈 힙에서 `peek()` / `pop()`은 `IndexError`를 발생시킵니다.
- 힌트: `_before(a, b)` 한 곳에서 min/max 방향을 결정하세요.

In [11]:
class BinaryHeap:
    def __init__(self, iterable=(), mode='min'):
        if mode not in ('min', 'max'):
            raise ValueError("mode must be 'min' or 'max'")
        self.mode = mode
        self._data = list(iterable)

        for i in range(len(self._data) // 2 - 1, -1, -1):
            self._sift_down(i)

    def __len__(self):
        return len(self._data)

    @property
    def data(self):
        return list(self._data)

    def _before(self, a, b):
        return a < b if self.mode == 'min' else a > b

    def _sift_up(self, i):
        d = self._data
        while i > 0:
            parent = (i - 1) // 2
            if not self._before(d[i], d[parent]):
                break
            d[i], d[parent] = d[parent], d[i]
            i = parent

    def _sift_down(self, i, end=None):
        d = self._data
        end = len(d) if end is None else end

        while True:
            best = i
            left, right = 2 * i + 1, 2 * i + 2

            if left < end and self._before(d[left], d[best]):
                best = left
            if right < end and self._before(d[right], d[best]):
                best = right
            if best == i:
                break

            d[i], d[best] = d[best], d[i]
            i = best

    def peek(self):
        if not self._data:
            raise IndexError('peek from empty heap')
        return self._data[0]

    def push(self, value):
        self._data.append(value)
        self._sift_up(len(self._data) - 1)

    def pop(self):
        if not self._data:
            raise IndexError('pop from empty heap')

        root = self._data[0]
        last = self._data.pop()
        if self._data:
            self._data[0] = last
            self._sift_down(0)
        return root

### 1-1. 공개 테스트 (BinaryHeap)

In [12]:
import random

# 01 min / max를 반복 pop해 정렬된 순서인지 확인
h = BinaryHeap([5, 3, 8, 1, 9, 2], mode='min')
assert [h.pop() for _ in range(6)] == [1, 2, 3, 5, 8, 9]
h = BinaryHeap([5, 3, 8, 1, 9, 2], mode='max')
assert [h.pop() for _ in range(6)] == [9, 8, 5, 3, 2, 1]

# 02 빈 힙의 peek / pop은 IndexError
e = BinaryHeap(mode='min')
for fn in (e.peek, e.pop):
    try:
        fn()
        assert False, '빈 힙인데 예외가 없음'
    except IndexError:
        pass

# 잘못된 mode는 ValueError
try:
    BinaryHeap(mode='mid')
    assert False
except ValueError:
    pass

# 03 단일 원소 · 중복 · 음수
h = BinaryHeap([-1, -1, 0, -5], mode='min')
assert h.peek() == -5 and len(h) == 4
assert [h.pop() for _ in range(4)] == [-5, -1, -1, 0]

# 04 각 push 뒤 불변식 확인 (부모 ≤ 자식 for min)
def is_min_heap(arr):
    return all(arr[(i - 1) // 2] <= arr[i] for i in range(1, len(arr)))

h = BinaryHeap(mode='min')
for v in [7, 2, 9, 4, 1]:
    h.push(v)
    assert is_min_heap(h.data)

# 05 연산을 섞어 리스트 오라클과 비교
random.seed(0)
h, oracle = BinaryHeap(mode='min'), []
for _ in range(300):
    if oracle and random.random() < 0.4:
        oracle.sort()
        assert h.pop() == oracle.pop(0)
    else:
        x = random.randint(-50, 50)
        h.push(x)
        oracle.append(x)
    assert len(h) == len(oracle)
print('BinaryHeap 공개 테스트 통과')

BinaryHeap 공개 테스트 통과


## 2. 제자리 오름차순 힙 정렬 `heapsort_inplace`

- 최대 힙을 사용합니다. 루트 `a[0]`와 `a[end-1]`를 교환하고 `end`를 줄입니다.
- 입력 리스트를 **직접 수정**하며 반환값은 `None`입니다.
- 추가 공간 O(1), 반복형 sift-down을 사용하세요(재귀·slicing·별도 결과 리스트 금지).
- 자식 인덱스 `c`는 반드시 `c < end`를 확인한 뒤 접근합니다.

In [13]:
def _sift_down_max(a, i, end):
    while True:
        largest = i
        left, right = 2 * i + 1, 2 * i + 2

        if left < end and a[left] > a[largest]:
            largest = left
        if right < end and a[right] > a[largest]:
            largest = right
        if largest == i:
            return

        a[i], a[largest] = a[largest], a[i]
        i = largest


def heapsort_inplace(a):
    n = len(a)

    for i in range(n // 2 - 1, -1, -1):
        _sift_down_max(a, i, n)

    for end in range(n - 1, 0, -1):
        a[0], a[end] = a[end], a[0]
        _sift_down_max(a, 0, end)

    return None

In [14]:
# 정렬 테스트
for case in [[], [1], [2, 1], [3, 3, 3], [5, -2, 9, 0, 5, 1], list(range(20, 0, -1))]:
    expected = sorted(case)
    ret = heapsort_inplace(case)
    assert ret is None, '반환값은 None이어야 합니다'
    assert case == expected, f'실패: {case}'

random.seed(1)
for _ in range(50):
    arr = [random.randint(-100, 100) for _ in range(random.randint(0, 60))]
    exp = sorted(arr)
    heapsort_inplace(arr)
    assert arr == exp
print('heapsort_inplace 테스트 통과')

heapsort_inplace 테스트 통과


**자기 점검:** `end - 1`을 자식으로 취급하면 어떤 버그가 생길까요? 길이 1인 힙에서 sift-down이 필요 없는 이유는 무엇인가요?

## 3. heapq 응용: 최대 힙 변환과 우선순위 큐

`heapq`는 **최소 힙**입니다. 숫자 최대 힙은 부호를 반전해 사용합니다.

In [15]:
import heapq

def max_from_list(values):
    heap = [-value for value in values]
    heapq.heapify(heap)
    return -heapq.heappop(heap)

assert max_from_list([4, 1, 7, 3]) == 7
assert max_from_list([-3, -9]) == -3
print('최댓값 테스트 통과')

최댓값 테스트 통과


### 3-1. heapreplace vs heappushpop

- `heappushpop(h, x)`: 먼저 x를 넣고, 최솟값을 반환합니다(x가 작으면 x를 그대로 반환하고 힙은 변하지 않음).
- `heapreplace(h, x)`: 먼저 루트를 꺼내 반환하고, 그 뒤 x를 넣습니다.

아래 셀의 빈칸을 채우고 결과를 비교하세요.

In [16]:
h1 = [3, 7, 5]
h2 = [3, 7, 5]
r1 = heapq.heappushpop(h1, 2)   # 기대 반환값: ?
r2 = heapq.heapreplace(h2, 2)   # 기대 반환값: ?
print('heappushpop 반환:', r1, '남은 힙:', h1)
print('heapreplace 반환:', r2, '남은 힙:', h2)
# TODO: 두 결과가 다른 이유를 주석으로 설명하세요.
assert r1 == 2 and sorted(h1) == [3, 5, 7]
assert r2 == 3 and sorted(h2) == [2, 5, 7]

heappushpop 반환: 2 남은 힙: [3, 7, 5]
heapreplace 반환: 3 남은 힙: [2, 7, 5]


### 3-2. 우선순위 큐 (동점은 먼저 들어온 작업부터)

튜플 `(priority, counter, task)`에서 `counter`는 삽입 순번입니다. `task`가 dict처럼 비교 불가능해도 `counter`에서 동점이 해결됩니다.

In [17]:
import itertools

class PriorityQueue:
    def __init__(self):
        self._heap = []
        self._counter = itertools.count()

    def __len__(self):
        return len(self._heap)

    def push(self, priority, task):
        heapq.heappush(self._heap, (priority, next(self._counter), task))

    def pop(self):
        if not self._heap:
            raise IndexError('pop from empty priority queue')

        priority, _, task = heapq.heappop(self._heap)
        return priority, task

q = PriorityQueue()
q.push(1, {'id': 'A'}); q.push(1, {'id': 'B'}); q.push(0, {'id': 'C'})
assert q.pop() == (0, {'id': 'C'})
assert q.pop()[1]['id'] == 'A'   # 동점은 FIFO
assert q.pop()[1]['id'] == 'B'
try:
    q.pop(); assert False
except IndexError:
    pass
print('우선순위 큐 테스트 통과')

우선순위 큐 테스트 통과


## 4. Top-k: 크기 k의 최소 힙 유지

- 힙 크기는 항상 k 이하입니다.
- `x > 루트`일 때만 루트를 제거하고 x를 넣습니다(같으면 교체하지 않음).
- 경계: `k <= 0`이면 `[]`, `k > n`이면 전체를 내림차순으로 반환합니다.

In [18]:
def top_k(iterable, k):
    if k <= 0:
        return []

    heap = []
    for value in iterable:
        if len(heap) < k:
            heapq.heappush(heap, value)
        elif value > heap[0]:
            heapq.heapreplace(heap, value)

    return sorted(heap, reverse=True)

assert top_k([5, 1, 9, 3, 7, 9], 3) == [9, 9, 7]
assert top_k([4, 4, 4], 2) == [4, 4]
assert top_k([1, 2], 0) == []
assert top_k([3, 1, 2], 10) == [3, 2, 1]
print('Top-k 테스트 통과')

Top-k 테스트 통과


## 5. 성능 실험 (선택 · 보고서용)

n = 500, 2000, 8000에서 무작위 / 정렬 / 역순 / 중복 입력을 만들고, 예열 후 3회 측정한 **중앙값**을 기록하세요. 측정 결과는 환경에 따라 달라지며, 정확성 검증은 측정 시간에서 제외합니다.

In [19]:
import time, statistics

def make_input(kind, n):
    if kind == 'random':
        return [random.randint(0, 10**6) for _ in range(n)]
    if kind == 'sorted':
        return list(range(n))
    if kind == 'reverse':
        return list(range(n, 0, -1))
    if kind == 'duplicates':
        return [random.randint(0, 9) for _ in range(n)]

def median_time(fn, data_factory, repeat=3):
    fn(data_factory())  # 예열
    times = []
    for _ in range(repeat):
        d = data_factory()
        t0 = time.perf_counter(); fn(d); times.append((time.perf_counter() - t0) * 1000)
    return statistics.median(times)

def heapq_sort(a):
    heapq.heapify(a)
    return [heapq.heappop(a) for _ in range(len(a))]


results = []
random.seed(2026)

for n in [500, 2000, 8000]:
    for kind in ['random', 'sorted', 'reverse', 'duplicates']:
        base = make_input(kind, n)
        expected = sorted(base)

        check = base.copy()
        heapsort_inplace(check)
        assert check == expected
        assert heapq_sort(base.copy()) == expected

        inplace_ms = median_time(heapsort_inplace, base.copy)
        heapq_ms = median_time(heapq_sort, base.copy)
        results.append((n, kind, inplace_ms, heapq_ms))

print('n | 입력 유형 | 제자리(ms) | heapq(ms)')
for n, kind, inplace_ms, heapq_ms in results:
    print(f'{n:4d} | {kind:10s} | {inplace_ms:10.4f} | {heapq_ms:8.4f}')

# 해석 질문 (주석으로 답하세요)
# 01 n이 4배가 될 때 시간은 어떻게 변하는가?
# 02 정렬·역순 입력에서 차이가 있는가?
# 03 코어 O(1) 공간과 래퍼의 O(n) 할당은 어떻게 다른가?
# 04 한 번의 실험으로 복잡도를 증명할 수 없는 이유는?
# 05 힙 정렬이 비안정 정렬인 이유는?

n | 입력 유형 | 제자리(ms) | heapq(ms)
 500 | random     |     1.2291 |   0.1324
 500 | sorted     |     1.2777 |   0.1324
 500 | reverse    |     1.0835 |   0.1337
 500 | duplicates |     1.0882 |   0.1359
2000 | random     |     6.5153 |   0.6569
2000 | sorted     |     6.4876 |   0.6337
2000 | reverse    |     5.9950 |   0.6492
2000 | duplicates |     5.7006 |   0.6194
8000 | random     |    38.4860 |   3.3103
8000 | sorted     |    31.9091 |   2.7819
8000 | reverse    |    28.6982 |   2.8511
8000 | duplicates |    27.0330 |   2.7087


## 6. 마무리 퀴즈 (주석으로 답하세요)

1. 완전이진트리와 포화이진트리는 어떤 점에서 다른가?
## 완전이진트리는 마지막 레벨을 제외한 모든 레벨이 꽉 차고, 마지막 레벨은 왼쪽부터 빈칸 없이 채운다. 포화이진트리는 마지막 레벨까지 꽉 찬다.
2. bottom-up heapify가 O(n log n)이 아니라 O(n)인 이유는?
## 리프는 이동하지 않고, 리프 바로 위의 노드는 최대 한 칸만 내려간다. 많이 내려갈 수 있는 위쪽 노드일수록 개수가 적으므로 모든 sift-down 비용의 합은 O(n)이다.
3. 제자리 오름차순 힙 정렬에 최대 힙을 쓰는 이유는?
## 아직 정렬하지 않은 원소 중 최댓값이 루트에 있기 때문이다. 이 값을 활성 구간의 맨 끝에 확정하는 작업을 반복하면 큰 값부터 오른쪽에 배치되어 전체 배열이 오름차순이 된다.
4. 작업끼리 비교할 수 없을 때 (priority, task) 튜플을 넣으면 어떤 문제가 생길까?
## 우선순위가 같으면 파이썬이 task의 순서를 비교하다가 TypeError를 낼 수 있다. (priority, counter, task)를 쓰면 동점일 때 삽입 순번으로 비교하므로 먼저 넣은 작업부터 나온다.
5. 힙 배열을 그대로 출력하면 정렬된 결과가 되는가?
## 아니다. 힙은 부모와 자식 사이의 순서만 보장한다. [1, 3, 2]는 올바른 최소 힙이지만 오름차순 배열은 아니다.